In [2]:
!pip install langchain-text-splitters transformers sentencepiece

   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/587.1 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/587.1 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/587.1 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/587.1 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/587.1 kB ? eta -:--:--
   --------------------------------- ---- 524.3/587.1 kB 279.5 kB/s eta 0:00:01
   ---------------------------------------- 587.1/587.1 kB 266.6 kB/s  0:00:02
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install pandas

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   -- ------------------------------------- 0.8/11.0 MB 817.9 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.0 MB 817.9 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.0 MB 817.9 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/11.0 MB 645.1 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/11.0 MB 645.1 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/11.0 MB 645.1 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/11.0 MB 645.1 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/11.0 MB 645.1 kB/s eta 0:00:16
   ---- ----------------------------------- 1.3/11.0 MB 453.3 kB/s eta 0:00:22
  


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Question 1

In [5]:
import os
import glob
import pandas as pd

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter


c:\Python_ML\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
DATASET_PATH = r"C:\Users\Prapti\OneDrive\Desktop\HPC\News\BBC News Summary\News Articles"

In [7]:
documents = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if os.path.isdir(category_path):

        files = glob.glob(os.path.join(category_path, "*.txt"))

        for file_path in files:

            try:
                with open(file_path, "r", encoding="utf-8") as file:
                    text = file.read()

            except UnicodeDecodeError:
                with open(file_path, "r", encoding="cp1252") as file:
                    text = file.read()

            documents.append({
                "filename": os.path.basename(file_path),
                "category": category,
                "content": text
            })

print("Total documents loaded:", len(documents))

Total documents loaded: 2225


In [8]:
print("First document:", documents[0]["filename"])
print("Category:", documents[0]["category"])
print("\nContent:\n")
print(documents[0]["content"][:500])

First document: 001.txt
Category: business

Content:

Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time W


In [9]:
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))
print("Tokenizer loaded successfully!")




Tokenizer loaded successfully!


In [10]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_sizes = [100, 150, 200]
chunk_overlap = 20

all_chunks = {}



In [11]:
for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=chunk_overlap,
        length_function=count_tokens,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = []

    for doc in documents:

        text_chunks = splitter.split_text(doc["content"])

        for i, chunk in enumerate(text_chunks):

            chunks.append({
                "chunk_id": len(chunks),
                "filename": doc["filename"],
                "category": doc["category"],
                "chunk_number": i + 1,
                "content": chunk,
                "token_count": count_tokens(chunk)
            })

    all_chunks[size] = chunks

    print(f"Chunk size: {size}")
    print(f"Total chunks: {len(chunks)}")
    print("-" * 40)

Token indices sequence length is longer than the specified maximum sequence length for this model (557 > 512). Running this sequence through the model will result in indexing errors


Chunk size: 100
Total chunks: 16421
----------------------------------------
Chunk size: 150
Total chunks: 11075
----------------------------------------
Chunk size: 200
Total chunks: 8096
----------------------------------------


In [12]:
chunk_comparison = pd.DataFrame({
    "Chunk Size": chunk_sizes,
    "Overlap": [chunk_overlap] * len(chunk_sizes),
    "Total Chunks": [
        len(all_chunks[size]) for size in chunk_sizes
    ]
})

display(chunk_comparison)


,Chunk Size,Overlap,Total Chunks
0,100,20,16421
1,150,20,11075
2,200,20,8096


In [14]:

sample_chunks = all_chunks[100]

print("Filename:", sample_chunks[0]["filename"])
print("Category:", sample_chunks[0]["category"])
print("Chunk number:", sample_chunks[0]["chunk_number"])
print("Token count:", sample_chunks[0]["token_count"])

print("\nChunk content:\n")
print(sample_chunks[0]["content"])

Filename: 001.txt
Category: business
Chunk number: 1
Token count: 46

Chunk content:

Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.


In [15]:
for size in chunk_sizes:

    print(f"\nChunk size: {size}")
    print("Number of chunks:", len(all_chunks[size]))

    print("First chunk tokens:",
          all_chunks[size][0]["token_count"])


Chunk size: 100
Number of chunks: 16421
First chunk tokens: 46

Chunk size: 150
Number of chunks: 11075
First chunk tokens: 125

Chunk size: 200
Number of chunks: 8096
First chunk tokens: 125


Question 2

In [14]:
!pip install sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
    --------------------------------------- 0.3/16.6 MB ? eta -:--:--
    --------------------------------------- 0.3/16.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.6 MB 645.7 kB/s eta 0:00:25
   - -------------------------------------- 0.5/16.6 MB 645.7 kB/s eta 0:00:25
   - -------------------------------------- 0.8/16.6 MB 588.4 kB/s eta 0:00:27
   -- ------------------------------------- 1.0/16.6 MB 637.3 kB/s eta 0:00:25
   -- ------------------------------------- 1.0/16.6 MB 637.3 kB/s eta 0:00:25
   --- ------------------------------------ 1.3/16.6 MB 633.2 kB/s eta 0:00:25
   --- ------------------------------------ 1.3/16.6 MB 633.2 kB/s eta 0:00:25
   --- ------------------------------------ 1.6/16.6 MB 590.7 kB/s eta 0:00:26
   --- ------------------------------------ 1.6/16.6 MB 590.7 kB/s eta 0:00:26
   --- ------------------------------------ 1.6/16.6 MB 590.7 kB/s eta 0:00:26



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer



In [16]:

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")



Embedding model loaded successfully!


In [18]:
embeddings_store = {}

for size in chunk_sizes:

    texts = [
        chunk["content"]
        for chunk in all_chunks[size]
    ]

    embeddings = embedding_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings_store[size] = embeddings

    print(f"\nChunk size: {size}")
    print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 257/257 [20:00<00:00,  4.67s/it]



Chunk size: 100
Embedding shape: (16421, 384)


Batches: 100%|██████████| 174/174 [17:00<00:00,  5.86s/it]



Chunk size: 150
Embedding shape: (11075, 384)


Batches: 100%|██████████| 127/127 [1:07:51<00:00, 32.06s/it] 



Chunk size: 200
Embedding shape: (8096, 384)


In [19]:
vector_stores = {}

dimension = 384

for size in chunk_sizes:

    index = faiss.IndexHNSWFlat(
        dimension,
        32,
        faiss.METRIC_INNER_PRODUCT
    )

    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 64

    index.add(
        embeddings_store[size].astype("float32")
    )

    vector_stores[size] = index

    print(f"\nChunk size: {size}")
    print("Total vectors:", index.ntotal)
    print("Vector dimension:", index.d)



Chunk size: 100
Total vectors: 16421
Vector dimension: 384

Chunk size: 150
Total vectors: 11075
Vector dimension: 384

Chunk size: 200
Total vectors: 8096
Vector dimension: 384


In [20]:

for size, index in vector_stores.items():

    print(
        f"Chunk size: {size} | "
        f"Vectors stored: {index.ntotal}"
    )



Chunk size: 100 | Vectors stored: 16421
Chunk size: 150 | Vectors stored: 11075
Chunk size: 200 | Vectors stored: 8096


Question 3

In [21]:
!pip install rank-bm25


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from rank_bm25 import BM25Okapi

bm25_stores = {}

for size in chunk_sizes:

    corpus = [
        tokenizer.tokenize(chunk["content"].lower())
        for chunk in all_chunks[size]
    ]

    bm25 = BM25Okapi(corpus)

    bm25_stores[size] = bm25

    print(f"BM25 index created for chunk size: {size}")
    print("Total documents:", len(corpus))
    print("-" * 40)

BM25 index created for chunk size: 100
Total documents: 16421
----------------------------------------
BM25 index created for chunk size: 150
Total documents: 11075
----------------------------------------
BM25 index created for chunk size: 200
Total documents: 8096
----------------------------------------


In [23]:
def dense_retrieval(question, chunk_size, top_k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    index = vector_stores[chunk_size]
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        chunk = all_chunks[chunk_size][idx]

        results.append({
            "filename": chunk["filename"],
            "category": chunk["category"],
            "content": chunk["content"],
            "score": float(score),
            "chunk_id": chunk["chunk_id"]
        })

    return results


In [24]:

question = "What caused Time Warner's quarterly profits to increase?"

results = dense_retrieval(
    question,
    chunk_size=100,
    top_k=5
)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("Filename:", result["filename"])
    print("Category:", result["category"])
    print("Similarity score:", round(result["score"], 4))
    print("Content:", result["content"][:300])
    print("-" * 60)


Result 1
Filename: 001.txt
Category: business
Similarity score: 0.7333
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
------------------------------------------------------------

Result 2
Filename: 001.txt
Category: business
Similarity score: 0.6361
Content: The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and
------------------------------------------------------------

Result 3
Filename: 001.txt
Category: business
Similarity score: 0.6342
Content: Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office f

In [25]:
def bm25_retrieval(question, chunk_size, top_k=5):

    query_tokens = tokenizer.tokenize(question.lower())

    bm25 = bm25_stores[chunk_size]

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:

        chunk = all_chunks[chunk_size][idx]

        results.append({
            "filename": chunk["filename"],
            "category": chunk["category"],
            "content": chunk["content"],
            "score": float(scores[idx]),
            "chunk_id": chunk["chunk_id"]
        })

    return results



In [26]:

question = "What caused Time Warner's quarterly profits to increase?"

bm25_results = bm25_retrieval(
    question,
    chunk_size=100,
    top_k=5
)

for i, result in enumerate(bm25_results, 1):

    print(f"\nResult {i}")
    print("Filename:", result["filename"])
    print("Category:", result["category"])
    print("BM25 score:", round(result["score"], 4))
    print("Content:", result["content"][:300])
    print("-" * 60)


Result 1
Filename: 001.txt
Category: business
BM25 score: 28.4119
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
------------------------------------------------------------

Result 2
Filename: 503.txt
Category: business
BM25 score: 23.809
Content: India's Maruti sees profits jump

India's biggest carmaker Maruti has reported a sharp increase in quarterly profit after a booming economy and low interest rates boosted demand.
------------------------------------------------------------

Result 3
Filename: 001.txt
Category: business
BM25 score: 20.9917
Content: Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exce


In [27]:
def hybrid_retrieval(question, chunk_size, top_k=5):

    bm25_results = bm25_retrieval(
        question, chunk_size, top_k=20
    )

    dense_results = dense_retrieval(
        question, chunk_size, top_k=20
    )

    rrf_scores = {}

    for rank, result in enumerate(bm25_results, 1):

        chunk_id = result["chunk_id"]

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)
        )

    for rank, result in enumerate(dense_results, 1):

        chunk_id = result["chunk_id"]

        rrf_scores[chunk_id] = (
            rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)
        )

    ranked_chunks = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for chunk_id, score in ranked_chunks[:top_k]:

        chunk = all_chunks[chunk_size][chunk_id]

        results.append({
            "filename": chunk["filename"],
            "category": chunk["category"],
            "content": chunk["content"],
            "score": score,
            "chunk_id": chunk_id
        })

    return results

In [28]:
question = "What caused Time Warner's quarterly profits to increase?"

hybrid_results = hybrid_retrieval(
    question,
    chunk_size=100,
    top_k=5
)

for i, result in enumerate(hybrid_results, 1):

    print(f"\nResult {i}")
    print("Filename:", result["filename"])
    print("Category:", result["category"])
    print("RRF Score:", round(result["score"], 5))
    print("Content:", result["content"][:300])
    print("-" * 60)


Result 1
Filename: 001.txt
Category: business
RRF Score: 0.03279
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
------------------------------------------------------------

Result 2
Filename: 001.txt
Category: business
RRF Score: 0.0315
Content: Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exce
------------------------------------------------------------

Result 3
Filename: 001.txt
Category: business
RRF Score: 0.0315
Content: Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and C

Question 4

In [29]:
question = "What caused Time Warner's quarterly profits to increase?"

candidate_chunks = hybrid_retrieval(
    question,
    chunk_size=100,
    top_k=20
)

print("Total candidate chunks:", len(candidate_chunks))



Total candidate chunks: 20


In [30]:

def bi_encoder_reranking(question, candidates, top_k=5):

    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    candidate_texts = [
        item["content"] for item in candidates
    ]

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = np.dot(
        candidate_embeddings,
        query_embedding
    )

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_indices:

        result = candidates[idx].copy()
        result["rerank_score"] = float(scores[idx])

        results.append(result)

    return results

In [31]:
bi_results = bi_encoder_reranking(
    question,
    candidate_chunks,
    top_k=5
)

for rank, result in enumerate(bi_results, 1):

    print(f"\nRank: {rank}")
    print("Filename:", result["filename"])
    print("Similarity:", round(result["rerank_score"], 4))
    print("Content:", result["content"][:250])





Rank: 1
Filename: 001.txt
Similarity: 0.7333
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

Rank: 2
Filename: 001.txt
Similarity: 0.6361
Content: The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off g

Rank: 3
Filename: 001.txt
Similarity: 0.6342
Content: Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final

Rank: 4
Filename: 001.txt
Similarity: 0.5715
Content: Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL,

In [32]:

from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-Encoder loaded successfully!")




'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: f27573c0-f368-49bb-b9ae-30e4d02dc60c)')' thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 25e25fe6-6625-48f3-b2fa-bec462f85c3d)')' thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modu

Cross-Encoder loaded successfully!


In [33]:
def cross_encoder_reranking(question, candidates, top_k=5):

    pairs = [
        [question, item["content"]]
        for item in candidates
    ]

    scores = cross_encoder.predict(
        pairs,
        show_progress_bar=True
    )

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_indices:

        result = candidates[idx].copy()
        result["rerank_score"] = float(scores[idx])

        results.append(result)

    return results


In [34]:
cross_results = cross_encoder_reranking(
    question,
    candidate_chunks,
    top_k=5
)

for rank, result in enumerate(cross_results, 1):

    print(f"\nRank: {rank}")
    print("Filename:", result["filename"])
    print("Cross-Encoder score:", round(result["rerank_score"], 4))
    print("Content:", result["content"][:250])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:05<00:00,  5.46s/it]


Rank: 1
Filename: 001.txt
Cross-Encoder score: 5.3492
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

Rank: 2
Filename: 001.txt
Cross-Encoder score: 4.0524
Content: Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final

Rank: 3
Filename: 001.txt
Cross-Encoder score: 3.1083
Content: Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, th

Rank: 4
Filename: 001.txt
Cross-Encoder score: 1.997
Content: The firm, which is now one of the biggest investors in Google, benefited

In [35]:

comparison = pd.DataFrame({
    "Without Reranking": [
        item["filename"] for item in candidate_chunks[:5]
    ],
    "Bi-Encoder": [
        item["filename"] for item in bi_results
    ],
    "Cross-Encoder": [
        item["filename"] for item in cross_results
    ]
})

comparison.index = range(1, 6)

display(comparison)

,Without Reranking,Bi-Encoder,Cross-Encoder
1,001.txt,001.txt,001.txt
2,001.txt,001.txt,001.txt
3,001.txt,001.txt,001.txt
4,001.txt,001.txt,001.txt
5,503.txt,001.txt,001.txt


Question 5

In [36]:
!pip install ollama


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import ollama

models = ollama.list()

print("Available Ollama models:")

for model in models.models:
    print(model.model)


Available Ollama models:
qwen2.5:0.5b
qwen3:8b
llama3.1:8b
llama3.2:latest


In [38]:
models = ollama.list()

for model in models.models:
    print(model.model)

qwen2.5:0.5b
qwen3:8b
llama3.1:8b
llama3.2:latest


In [39]:
response = ollama.chat(
    model="qwen2.5:0.5b",
    messages=[
        {
            "role": "user",
            "content": "Explain Retrieval-Augmented Generation in two simple sentences."
        }
    ]
)

print(response.message.content)

Retrieval-Augmented Generation (RAG) is a natural language processing technique that improves the quality of text by integrating the knowledge and context derived from the surrounding text. RAG uses a set of predefined data sources (e.g., Wikipedia, social media, news articles) to generate text that complements and enhances the original text. This approach allows the system to better understand and generate natural, coherent text that reflects the context and relationships between the words in the input.


In [40]:
def generate_rag_answer(question, retrieved_chunks):

    context = ""

    for i, chunk in enumerate(retrieved_chunks, 1):
        context += f"""
        Source: {chunk['filename']}
        Article content:
        {chunk['content']}

        """
    prompt = f"""
    Answer the question using ONLY the information in the context below.

    Rules:
    1. Do not use outside knowledge.
    2. Do not invent facts.
    3. If the answer is not available in the context, say:
       "The provided articles do not contain enough information to answer this question."
    4. Give a clear and concise answer.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = ollama.chat(
        model="qwen2.5:0.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.message.content

    # Extract unique source filenames
    sources = list(dict.fromkeys(
        chunk["filename"] for chunk in retrieved_chunks
    ))

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

print("RAG answer-generation function created successfully!")

RAG answer-generation function created successfully!


In [41]:
question = "What caused Time Warner's quarterly profits to increase?"

retrieved_chunks = hybrid_retrieval(
    question,
    chunk_size=100,
    top_k=20
)

reranked_chunks = cross_encoder_reranking(
    question,
    retrieved_chunks,
    top_k=5
)

result = generate_rag_answer(
    question,
    reranked_chunks
)

print("QUESTION:")
print(result["question"])

print("\nGENERATED ANSWER:")
print(result["answer"])

print("\nSOURCE FILENAMES:")
for source in result["sources"]:
    print("-", source)

Batches: 100%|██████████| 1/1 [00:04<00:00,  4.93s/it]


QUESTION:
What caused Time Warner's quarterly profits to increase?

GENERATED ANSWER:
Time Warner's quarterly profits increased due to the following reasons:

1. The fourth quarter saw a slight improvement in performance compared to the previous three quarters.
2. The film division saw its profits decrease by 27% to $284 million, which was helped by box-office disappointments such as Alexander and Catwoman.
3. The full-year profits for Time Warner increased to $3.36 billion, up 27% from its 2003 performance.
4. The company's overall operating profit growth for the year was around 5%, which also helped to offset some losses in the film division.
5. Time Warner also saw an increase in its revenue, reaching $42.09 billion, compared to the $10.9 billion in 2003.

SOURCE FILENAMES:
- 001.txt


In [42]:
unsupported_question = "What was the exact name of the CEO's childhood school?"

test_chunks = hybrid_retrieval(
    unsupported_question,
    chunk_size=100,
    top_k=5
)
unsupported_result = generate_rag_answer(
    unsupported_question,
    test_chunks
)
print("QUESTION:")
print(unsupported_result["question"])

print("\nGENERATED ANSWER:")
print(unsupported_result["answer"])

print("\nSOURCE FILENAMES:")
for source in unsupported_result["sources"]:
    print("-", source)

QUESTION:
What was the exact name of the CEO's childhood school?

GENERATED ANSWER:
The provided articles do not contain enough information to answer the question about the exact name of the CEO's childhood school.

SOURCE FILENAMES:
- 232.txt
- 123.txt
- 111.txt
- 219.txt
- 293.txt


Question 6

In [43]:
evaluation_questions = [
    "What caused Time Warner's quarterly profits to increase?",
    "How did AOL's advertising revenue affect its profits?",
    "What happened to Time Warner's film division profits?",
    "What were Time Warner's annual profits?",
    "What factors contributed to the growth of Time Warner?",
    "What were the main challenges faced by the company?",
    "How did internet advertising affect company performance?",
    "What financial results were reported by the company?",
    "What changes were observed in the company's revenue?",
    "What were the major business developments reported in the article?"
]

print("Total evaluation questions:", len(evaluation_questions))

for i, question in enumerate(evaluation_questions, 1):
    print(f"{i}. {question}")

Total evaluation questions: 10
1. What caused Time Warner's quarterly profits to increase?
2. How did AOL's advertising revenue affect its profits?
3. What happened to Time Warner's film division profits?
4. What were Time Warner's annual profits?
5. What factors contributed to the growth of Time Warner?
6. What were the main challenges faced by the company?
7. How did internet advertising affect company performance?
8. What financial results were reported by the company?
9. What changes were observed in the company's revenue?
10. What were the major business developments reported in the article?


In [44]:
for i, question in enumerate(evaluation_questions, 1):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}: {question}")
    print("=" * 80)

    results = hybrid_retrieval(
        question,
        chunk_size=100,
        top_k=3
    )

    for rank, result in enumerate(results, 1):
        print(f"\nRank {rank}")
        print("Source:", result["filename"])
        print("Category:", result["category"])
        print("Content:", result["content"][:300])


QUESTION 1: What caused Time Warner's quarterly profits to increase?

Rank 1
Source: 001.txt
Category: business
Content: Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

Rank 2
Source: 001.txt
Category: business
Content: Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exce

Rank 3
Source: 001.txt
Category: business
Content: Time Warner's fourth quarter profits were slightly better than analysts' expectations. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted res



In [45]:
retrieval_summary = []
for i, question in enumerate(evaluation_questions, 1):

    results = hybrid_retrieval(
        question,
        chunk_size=100,
        top_k=3
    )
    retrieval_summary.append({
        "Question No.": i,
        "Question": question,
        "Top Source": results[0]["filename"],
        "Category": results[0]["category"],
        "Top Score": round(results[0]["score"], 4)
    })
retrieval_summary_df = pd.DataFrame(retrieval_summary)

display(retrieval_summary_df)

,Question No.,Question,Top Source,Category,Top Score
0,1,What caused Time Warner's quarterly profits to...,001.txt,business,0.0328
1,2,How did AOL's advertising revenue affect its p...,001.txt,business,0.0323
2,3,What happened to Time Warner's film division p...,001.txt,business,0.0328
3,4,What were Time Warner's annual profits?,001.txt,business,0.0325
4,5,What factors contributed to the growth of Time...,155.txt,tech,0.0164
5,6,What were the main challenges faced by the com...,324.txt,business,0.0265
6,7,How did internet advertising affect company pe...,001.txt,business,0.0295
7,8,What financial results were reported by the co...,467.txt,business,0.0311
8,9,What changes were observed in the company's re...,467.txt,business,0.0273
9,10,What were the major business developments repo...,482.txt,business,0.0323


In [46]:
for category in ["business", "entertainment", "politics", "sport", "tech"]:

    print("\n" + "=" * 60)
    print("CATEGORY:", category.upper())
    print("=" * 60)

    category_docs = [
        doc for doc in documents
        if doc["category"] == category
    ]

    for doc in category_docs[:5]:
        first_line = doc["content"].strip().split("\n")[0]
        print(doc["filename"], ":", first_line)


CATEGORY: BUSINESS
001.txt : Ad sales boost Time Warner profit
002.txt : Dollar gains on Greenspan speech
003.txt : Yukos unit buyer faces loan claim
004.txt : High fuel prices hit BA's profits
005.txt : Pernod takeover talk lifts Domecq

CATEGORY: ENTERTAINMENT
001.txt : Gallery unveils interactive tree
002.txt : Jarre joins fairytale celebration
003.txt : Musical treatment for Capra film
004.txt : Richard and Judy choose top books
005.txt : Poppins musical gets flying start

CATEGORY: POLITICS
001.txt : Labour plans maternity pay rise
002.txt : Watchdog probes e-mail deletions
003.txt : Hewitt decries 'career sexism'
004.txt : Labour chooses Manchester
005.txt : Brown ally rejects Budget spree

CATEGORY: SPORT
001.txt : Claxton hunting first major medal
002.txt : O'Sullivan could run in Worlds
003.txt : Greene sets sights on world title
004.txt : IAAF launches fight against drugs
005.txt : Dibaba breaks 5,000m world record

CATEGORY: TECH
001.txt : Ink helps drive democracy in Asia


In [47]:
for category in ["business", "entertainment", "politics", "sport", "tech"]:

    category_docs = [
        doc for doc in documents
        if doc["category"] == category
    ]

    print("\n", category.upper())

    for doc in category_docs[:2]:
        title = doc["content"].strip().split("\n")[0]
        print(f"{doc['filename']} : {title}")
        


 BUSINESS
001.txt : Ad sales boost Time Warner profit
002.txt : Dollar gains on Greenspan speech

 ENTERTAINMENT
001.txt : Gallery unveils interactive tree
002.txt : Jarre joins fairytale celebration

 POLITICS
001.txt : Labour plans maternity pay rise
002.txt : Watchdog probes e-mail deletions

 SPORT
001.txt : Claxton hunting first major medal
002.txt : O'Sullivan could run in Worlds

 TECH
001.txt : Ink helps drive democracy in Asia
002.txt : China net cafe culture crackdown


In [48]:
evaluation_questions = [
    "How did advertising sales affect Time Warner's profits?",
    "What caused the dollar to gain value following Greenspan's speech?",

    "What is the interactive tree exhibition about?",
    "What role did Jean Michel Jarre play in the fairytale celebration?",

    "What changes were planned for maternity pay by the Labour government?",
    "Why was the watchdog investigating the deletion of emails?",

    "What major medal was Claxton aiming to win?",
    "What were the chances of O'Sullivan participating in the World Championships?",

    "How can ink technology help drive democracy in Asia?",
    "Why was China cracking down on internet cafes?"
]

print("Total evaluation questions:", len(evaluation_questions))

for i, question in enumerate(evaluation_questions, 1):
    print(f"{i}. {question}")


Total evaluation questions: 10
1. How did advertising sales affect Time Warner's profits?
2. What caused the dollar to gain value following Greenspan's speech?
3. What is the interactive tree exhibition about?
4. What role did Jean Michel Jarre play in the fairytale celebration?
5. What changes were planned for maternity pay by the Labour government?
6. Why was the watchdog investigating the deletion of emails?
7. What major medal was Claxton aiming to win?
8. What were the chances of O'Sullivan participating in the World Championships?
9. How can ink technology help drive democracy in Asia?
10. Why was China cracking down on internet cafes?


In [49]:
balanced_retrieval_results = []

for i, question in enumerate(evaluation_questions, 1):

    results = hybrid_retrieval(
        question,
        chunk_size=100,
        top_k=5
    )

    balanced_retrieval_results.append({
        "Question No.": i,
        "Question": question,
        "Expected Category": [
            "business", "business",
            "entertainment", "entertainment",
            "politics", "politics",
            "sport", "sport",
            "tech", "tech"
        ][i-1],
        "Retrieved Source": results[0]["filename"],
        "Retrieved Category": results[0]["category"],
        "Score": round(results[0]["score"], 4)
    })

balanced_retrieval_df = pd.DataFrame(balanced_retrieval_results)

display(balanced_retrieval_df)


,Question No.,Question,Expected Category,Retrieved Source,Retrieved Category,Score
0,1,How did advertising sales affect Time Warner's...,business,001.txt,business,0.0328
1,2,What caused the dollar to gain value following...,business,002.txt,business,0.0325
2,3,What is the interactive tree exhibition about?,entertainment,001.txt,entertainment,0.0328
3,4,What role did Jean Michel Jarre play in the fa...,entertainment,002.txt,entertainment,0.0328
4,5,What changes were planned for maternity pay by...,politics,001.txt,politics,0.0328
5,6,Why was the watchdog investigating the deletio...,politics,002.txt,politics,0.0328
6,7,What major medal was Claxton aiming to win?,sport,001.txt,sport,0.0325
7,8,What were the chances of O'Sullivan participat...,sport,002.txt,sport,0.0323
8,9,How can ink technology help drive democracy in...,tech,001.txt,tech,0.0328
9,10,Why was China cracking down on internet cafes?,tech,002.txt,tech,0.0318


In [50]:
methods = {
    "BM25": bm25_retrieval,
    "Dense": dense_retrieval,
    "Hybrid": hybrid_retrieval
}

expected_categories = [
    "business", "business",
    "entertainment", "entertainment",
    "politics", "politics",
    "sport", "sport",
    "tech", "tech"
]

experiment_results = []

for chunk_size in chunk_sizes:

    for method_name, retrieval_function in methods.items():

        print(
            f"Testing Chunk Size: {chunk_size}, "
            f"Method: {method_name}"
        )

        for i, question in enumerate(evaluation_questions):

            results = retrieval_function(
                question,
                chunk_size=chunk_size,
                top_k=5
            )

            top_result = results[0]

            experiment_results.append({
                "Question No.": i + 1,
                "Chunk Size": chunk_size,
                "Retrieval Method": method_name,
                "Expected Category": expected_categories[i],
                "Retrieved Category": top_result["category"],
                "Source": top_result["filename"],
                "Top Score": round(top_result["score"], 4),
                "Category Match": (
                    top_result["category"] == expected_categories[i]
                )
            })

print("\nRetrieval experiments completed!")

Testing Chunk Size: 100, Method: BM25
Testing Chunk Size: 100, Method: Dense
Testing Chunk Size: 100, Method: Hybrid
Testing Chunk Size: 150, Method: BM25
Testing Chunk Size: 150, Method: Dense
Testing Chunk Size: 150, Method: Hybrid
Testing Chunk Size: 200, Method: BM25
Testing Chunk Size: 200, Method: Dense
Testing Chunk Size: 200, Method: Hybrid

Retrieval experiments completed!


In [51]:
experiment_df = pd.DataFrame(experiment_results)

retrieval_comparison = (
    experiment_df
    .groupby(["Chunk Size", "Retrieval Method"])
    .agg(
        Total_Questions=("Question No.", "count"),
        Category_Matches=("Category Match", "sum"),
        Average_Top_Score=("Top Score", "mean")
    )
    .reset_index()
)

retrieval_comparison["Category Match (%)"] = (
    retrieval_comparison["Category_Matches"]
    / retrieval_comparison["Total_Questions"] * 100
).round(2)

display(retrieval_comparison)

,Chunk Size,Retrieval Method,Total_Questions,Category_Matches,Average_Top_Score,Category Match (%)
0,100,BM25,10,10,35.74936,100.0
1,100,Dense,10,10,0.69867,100.0
2,100,Hybrid,10,10,0.03259,100.0
3,150,BM25,10,10,36.67872,100.0
4,150,Dense,10,10,0.68974,100.0
5,150,Hybrid,10,10,0.03269,100.0
6,200,BM25,10,10,37.98943,100.0
7,200,Dense,10,10,0.67445,100.0
8,200,Hybrid,10,10,0.03270,100.0


In [52]:
reranking_results = []

for i, question in enumerate(evaluation_questions):

    candidates = hybrid_retrieval(
        question,
        chunk_size=100,
        top_k=20
    )

    original_results = candidates[:5]

    bi_results = bi_encoder_reranking(
        question,
        candidates,
        top_k=5
    )

    cross_results = cross_encoder_reranking(
        question,
        candidates,
        top_k=5
    )

    for method, results in [
        ("Without Reranking", original_results),
        ("Bi-Encoder", bi_results),
        ("Cross-Encoder", cross_results)
    ]:

        top_result = results[0]

        reranking_results.append({
            "Question No.": i + 1,
            "Question": question,
            "Method": method,
            "Top Source": top_result["filename"],
            "Category": top_result["category"],
            "Top Chunk": top_result["content"][:150]
        })

print("Reranking experiments completed!")


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]

Reranking experiments completed!


In [53]:
reranking_df = pd.DataFrame(reranking_results)

display(
    reranking_df[
        ["Question No.", "Method", "Top Source", "Category"]
    ]
)

,Question No.,Method,Top Source,Category
0,1,Without Reranking,001.txt,business
1,1,Bi-Encoder,001.txt,business
2,1,Cross-Encoder,001.txt,business
3,2,Without Reranking,002.txt,business
4,2,Bi-Encoder,002.txt,business
5,2,Cross-Encoder,002.txt,business
6,3,Without Reranking,001.txt,entertainment
7,3,Bi-Encoder,001.txt,entertainment
8,3,Cross-Encoder,001.txt,entertainment
9,4,Without Reranking,002.txt,entertainment


In [54]:

sample_results = []

for i, question in enumerate(evaluation_questions[:5], 1):

    print(f"\nGenerating answer for Question {i}...")

  
    retrieved_chunks = hybrid_retrieval(
        question,
        chunk_size=100,
        top_k=20
    )

    reranked_chunks = cross_encoder_reranking(
        question,
        retrieved_chunks,
        top_k=5
    )

    result = generate_rag_answer(
        question,
        reranked_chunks
    )

    sample_results.append({
        "Question No.": i,
        "Question": result["question"],
        "Generated Answer": result["answer"],
        "Source Files": ", ".join(result["sources"])
    })

    print("Answer generated successfully!")

sample_answers_df = pd.DataFrame(sample_results)

pd.set_option("display.max_colwidth", 300)

display(sample_answers_df)



Generating answer for Question 1...


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.06s/it]


Answer generated successfully!

Generating answer for Question 2...


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.11s/it]


Answer generated successfully!

Generating answer for Question 3...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.46s/it]


... (2 outputs left out)

Answer generated successfully!

Generating answer for Question 5...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.19s/it]


Answer generated successfully!


,Question No.,Question,Generated Answer,Source Files
0,1,How did advertising sales affect Time Warner's profits?,"Time Warner's profits were boosted by advertising sales. According to the context, Time Warner's fourth quarter profits rose 2% to $11.1bn from $10.9bn. Its profits were boosted by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.","001.txt, 035.txt"
1,2,What caused the dollar to gain value following Greenspan's speech?,"The dollar gained value following Greenspan's speech because it was highlighted as a factor that may help to reduce the US trade deficit, which Alan Greenspan earlier expressed concern about during his speech in London.","002.txt, 273.txt"
2,3,What is the interactive tree exhibition about?,The interactive tree exhibition at London's Tate Britain art gallery involves a spruce tree that can receive text messages from visitors. Visitors can take photos with their phones and upload them to the web for the tree to display. The spruce tree is equipped with an antenna that can receive Bl...,"001.txt, 005.txt, 115.txt, 046.txt"
3,4,What role did Jean Michel Jarre play in the fairytale celebration?,"In the fairytale celebration, Jean Michel Jarre played Ed Wood. He played a role under Burton, Ed Wood, in the 1994 film What's Eating Gilbert Grape.","002.txt, 316.txt, 045.txt, 149.txt"
4,5,What changes were planned for maternity pay by the Labour government?,"The Labour government plans to extend the paid maternity pay period to nine months starting from 2007, with an additional £1,400 increase over the next nine years. Additionally, maternity pay will be given to fathers and flexible working rights will be extended to carers and parents of older chi...","001.txt, 340.txt, 003.txt"
